# 03 — Baselines

How well can each protein (CLR) be predicted, and how much of that comes from cell-type identity?

| # | Model | Input |
|---|---|---|
| 0 | Global mean | nothing (constant per protein) |
| 1 | Cell-type mean | **the annotated cell-type label** (not RNA) |
| 2 | Own gene | RNA of the protein's mapped gene(s): `y = a + b·x` per protein |
| 3a | Ridge on PCA | PCA of 2,000 HVGs |
| 3b | Ridge on HVG | 2,000 z-scored HVGs |

**Rules**
* Random cell-level split 70/15/15, stratified by cell type, saved once to `results/splits/random_split.csv.gz`.
* Everything learned from data (means, slopes, HVG selection, scaling, PCA, ridge weights) is fit on
  **training cells only**.
* Hyperparameters are chosen on validation. The selection metric, fixed beforehand, is the mean
  validation Pearson (all cells) over all 134 proteins.
* **Test cells are not evaluated in this notebook.** They are used once, after all models (including
  Phase 4) are frozen.
* Headline metrics use the 110-protein **primary** set. Secondary (7) and unavailable (17) proteins are
  reported separately.
* Undefined correlations stay NaN and are never counted as 0: the global mean has no Pearson at all,
  and the cell-type mean has no within-type Pearson.
* Validation numbers for ridge are slightly optimistic, because its hyperparameters were picked on
  the same cells. The test set will give an unbiased number.

In [ ]:
import os, sys, time, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import h5py
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt

from citeseq.paths import CITE_H5AD, REPO_ROOT, RESULTS_DIR, FIGURES_DIR, PROCESSED_DIR
from citeseq.io import load_cite, _read_dataframe
from citeseq.correlation import load_protein_gene_map, mapped_log_expression, pearson_columns
from citeseq.splits import make_random_split, save_split, load_split
from citeseq.baselines import GlobalMean, CellTypeMean, OwnGeneLinear, RidgePath, HVGPCAFeatures
from citeseq.metrics import evaluate, headline
from citeseq import plotting as P

SEED = 0
MIN_CELLS = 50
N_HVG, MAX_PCS = 2000, 200
PCS_GRID = [50, 100, 200]
ALPHA_PCA = [float(a) for a in np.logspace(-2, 4, 13)]
ALPHA_HVG = [float(a) for a in np.logspace(0, 5, 11)]
SUBSET = int(os.environ.get("CITESEQ_SUBSET", 0)) or None
OUT_FIG = FIGURES_DIR if SUBSET is None else PROCESSED_DIR / "test_figures"
OUT_FIG.mkdir(parents=True, exist_ok=True)
SPLIT_PATH = RESULTS_DIR / "splits/random_split.csv.gz" if SUBSET is None else PROCESSED_DIR / "test_split.csv.gz"
P.set_style()
pd.set_option("display.float_format", "{:.3f}".format)
print("subset:", SUBSET)

## 1. Load data and the split

In [ ]:
with h5py.File(CITE_H5AD) as f:
    var = _read_dataframe(f["var"])
mask = None
if SUBSET:
    mask = np.zeros(90_261, bool)
    mask[np.random.default_rng(SEED).choice(mask.size, SUBSET, replace=False)] = True
rna = load_cite(CITE_H5AD, "layers/counts", cell_mask=mask, var_mask=(var["feature_types"] == "GEX").to_numpy())
adt = load_cite(CITE_H5AD, "X", cell_mask=mask, var_mask=(var["feature_types"] == "ADT").to_numpy())
proteins = list(adt.var_names)
pmap = load_protein_gene_map(REPO_ROOT / "resources/protein_gene_map.csv", rna.var_names).set_index("protein").loc[proteins]
SETS = pmap["analysis_set"]
primary = SETS.index[SETS == "primary"].tolist()
print(rna.shape, adt.shape, SETS.value_counts().to_dict())

if SPLIT_PATH.exists() and SUBSET is None:
    split = load_split(SPLIT_PATH, rna.obs_names)
    print("loaded existing split:", SPLIT_PATH.name)
else:
    split = make_random_split(rna.obs, seed=SEED)
    save_split(split, SPLIT_PATH)
    print("created and saved split:", SPLIT_PATH)
print(split.value_counts().to_string())
print("min cells of a cell type per part:", pd.crosstab(rna.obs["cell_type"], split).min().to_dict())

## 2. Inputs

* Own-gene RNA: summed raw counts of the mapped gene(s) -> log1p CP10k (NaN column for proteins without
  a measured gene).
* All genes: log1p CP10k. This is per cell, so no information is shared between cells.
* Only training and validation cells are taken out; **test cells are never indexed below.**

In [ ]:
total = np.asarray(rna.X.sum(axis=1)).ravel()
usable = pmap["usable"].to_numpy()
R = np.full((rna.n_obs, len(proteins)), np.nan, dtype=np.float32)
R[:, usable] = mapped_log_expression(rna.X, rna.var_names, total, pmap.loc[usable, "gene_list"].tolist())
sc.pp.normalize_total(rna, target_sum=1e4)
sc.pp.log1p(rna)

tr = (split == "train").to_numpy()
va = (split == "val").to_numpy()
print(f"train {tr.sum():,} | val {va.sum():,} | test {(split == 'test').sum():,} (not used here)")
X_tr, X_va = rna.X[tr], rna.X[va]
obs_tr = rna.obs.loc[tr]
ct_tr = rna.obs["cell_type"].astype(str).to_numpy()[tr]
ct_va = rna.obs["cell_type"].astype(str).to_numpy()[va]
Y_all = adt.X.toarray()
Y_tr, Y_va = Y_all[tr], Y_all[va]
R_tr, R_va = R[tr], R[va]
gene_names = np.asarray(rna.var_names)
del rna, adt, Y_all, R

## 3. Train-only features: HVG, scaling, PCA

In [ ]:
t0 = time.time()
feats = HVGPCAFeatures(n_top_genes=N_HVG, max_pcs=MAX_PCS, seed=SEED).fit(X_tr, obs_tr, gene_names)
Z_tr, Z_va = feats.transform_pca(X_tr, MAX_PCS), feats.transform_pca(X_va, MAX_PCS)
H_tr, H_va = feats.transform_hvg(X_tr), feats.transform_hvg(X_va)
print(f"features in {time.time() - t0:.0f}s | HVGs: {feats.hvg_mask_.sum()} | "
      f"PCA variance explained by {MAX_PCS} PCs: {feats.pca_.explained_variance_ratio_.sum():.3f}")
own_genes_in_hvg = pmap.loc[usable, "gene_list"].explode().isin(feats.hvg_names_).mean()
print(f"share of mapped genes that are among the HVGs: {own_genes_in_hvg:.1%}")

## 4. Models 0-2 (no hyperparameters)

In [ ]:
pred = {}
pred["0 global mean"] = GlobalMean().fit(Y_tr).predict(len(Y_va))
ctm = CellTypeMean().fit(Y_tr, ct_tr)
pred["1 cell-type mean (labels)"] = ctm.predict(ct_va)
print("cell types in validation unseen in training:", ctm.n_fallback_)
own = OwnGeneLinear().fit(R_tr, Y_tr)
pred["2 own gene"] = own.predict(R_va)
print("own-gene slopes < 0 (primary set):", int((own.slope_[SETS.to_numpy() == 'primary'] < 0).sum()))

## 5. Ridge: hyperparameters on validation

Selection metric (fixed beforehand): mean validation Pearson over all 134 proteins.

In [ ]:
def sel_metric(p):
    return float(np.nanmean(pearson_columns(p, Y_va)))

grid = []
t0 = time.time()
for n in PCS_GRID:
    rp = RidgePath().fit(Z_tr[:, :n], Y_tr, ALPHA_PCA)
    for a in ALPHA_PCA:
        grid.append({"model": "3a ridge PCA", "n_pcs": n, "alpha": a, "val_mean_pearson": sel_metric(rp.predict(Z_va[:, :n], a))})
rp_h = RidgePath().fit(H_tr, Y_tr, ALPHA_HVG)
for a in ALPHA_HVG:
    grid.append({"model": "3b ridge HVG", "n_pcs": np.nan, "alpha": a, "val_mean_pearson": sel_metric(rp_h.predict(H_va, a))})
grid = pd.DataFrame(grid)
print(f"grid search in {time.time() - t0:.0f}s")
best = grid.loc[grid.groupby("model")["val_mean_pearson"].idxmax()].set_index("model")
best

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4), sharey=True)
ax = axes[0]
styles = {50: ":", 100: "--", 200: "-"}
for n in PCS_GRID:
    g = grid[(grid["model"] == "3a ridge PCA") & (grid["n_pcs"] == n)]
    ax.plot(g["alpha"], g["val_mean_pearson"], color=P.BLUE, ls=styles[n], lw=2, label=f"{n} PCs")
ax.set_xscale("log"); ax.set_xlabel("alpha"); ax.set_ylabel("mean val Pearson (134 proteins)")
ax.set_title("3a ridge on PCA"); ax.legend(frameon=False, fontsize=8)
ax = axes[1]
g = grid[grid["model"] == "3b ridge HVG"]
ax.plot(g["alpha"], g["val_mean_pearson"], color=P.BLUE, lw=2, marker="o", ms=3)
ax.set_xscale("log"); ax.set_xlabel("alpha"); ax.set_title("3b ridge on 2,000 HVGs")
fig.tight_layout()
fig.savefig(OUT_FIG / "phase3_ridge_selection.png")

In [ ]:
b = best.loc["3a ridge PCA"]
n_best, a_best = int(b["n_pcs"]), float(b["alpha"])
pred["3a ridge PCA"] = RidgePath().fit(Z_tr[:, :n_best], Y_tr, [a_best]).predict(Z_va[:, :n_best], a_best)
a_hvg = float(best.loc["3b ridge HVG", "alpha"])
pred["3b ridge HVG"] = rp_h.predict(H_va, a_hvg)

frozen = {
    "split_file": "results/splits/random_split.csv.gz", "seed": SEED,
    "features": {"hvg": N_HVG, "hvg_flavor": "seurat", "hvg_batch_key": "batch",
                 "pca_solver": "covariance_eigh", "max_pcs": MAX_PCS},
    "selection_metric": "mean validation Pearson over all 134 proteins",
    "3a_ridge_pca": {"n_pcs": n_best, "alpha": a_best},
    "3b_ridge_hvg": {"alpha": a_hvg},
    "min_cells_within_type": MIN_CELLS,
}
if SUBSET is None:
    (RESULTS_DIR / "phase3_frozen_configs.json").write_text(json.dumps(frozen, indent=2))
frozen

## 6. Validation results

In [ ]:
per = {m: evaluate(Y_va, p, ct_va, proteins, MIN_CELLS) for m, p in pred.items()}
long = pd.concat({m: d.assign(analysis_set=SETS) for m, d in per.items()}, names=["model"]).reset_index()
if SUBSET is None:
    long.to_csv(RESULTS_DIR / "phase3_val_per_protein.csv", index=False)

def fmt(h, m):
    r = h.loc[m]
    if r["n_defined"] == 0:
        return "undefined"
    return f"{r['median']:.3f} [{r['q25']:.3f}-{r['q75']:.3f}]"

def table(prots):
    rows = {}
    for m, d in per.items():
        h = headline(d, prots)
        rows[m] = {"Pearson, all cells": fmt(h, "pearson_all"),
                   "within type, weighted": fmt(h, "within_w"),
                   "within type, unweighted": fmt(h, "within_unw"),
                   "RMSE": fmt(h, "rmse"),
                   "proteins with defined Pearson": f"{int(h.loc['pearson_all', 'n_defined'])}/{len(prots)}"}
    return pd.DataFrame(rows).T

n_types_val = pd.Series(ct_va).value_counts().ge(MIN_CELLS).sum()
print(f"PRIMARY set ({len(primary)} proteins), validation, median [IQR] across proteins; "
      f"within-type over {n_types_val} cell types with >= {MIN_CELLS} validation cells")
table(primary)

In [ ]:
print("SECONDARY set (separate)")
display(table(SETS.index[SETS == "secondary"].tolist()))
print("UNAVAILABLE set (no measured gene: own-gene model cannot predict them)")
table(SETS.index[SETS == "unavailable"].tolist())

In [ ]:
# How often is a within-type r undefined for a model that should have one? (primary set)
for m in ["2 own gene", "3a ridge PCA", "3b ridge HVG"]:
    d = per[m].loc[primary]
    print(f"{m:<16} cell types used per protein: {int(d['n_types'].median())}, "
          f"proteins with >= 1 undefined type: {(d['n_types_nan'] > 0).sum()}, "
          f"undefined (protein, type) pairs: {int(d['n_types_nan'].sum())}")

## 7. What do multi-gene models add over the own gene and over cell-type identity?

In [ ]:
best_ridge = max(["3a ridge PCA", "3b ridge HVG"], key=lambda m: best.loc[m, "val_mean_pearson"])
print("better ridge by the selection metric:", best_ridge)
own_p, rid_p, ctm_p = per["2 own gene"].loc[primary], per[best_ridge].loc[primary], per["1 cell-type mean (labels)"].loc[primary]
for metric in ["pearson_all", "within_w"]:
    d = rid_p[metric] - own_p[metric]
    print(f"{metric:<12} ridge - own gene: median {d.median():+.3f}; ridge better for {(d > 0).sum()} of {d.notna().sum()}")
d = rid_p["pearson_all"] - ctm_p["pearson_all"]
print(f"pearson_all  ridge - cell-type mean: median {d.median():+.3f}; ridge better for {(d > 0).sum()} of {d.notna().sum()}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
order = list(per)
for ax, metric, title in zip(axes, ["pearson_all", "within_w"], ["Pearson, all validation cells", "Pearson within cell type (weighted)"]):
    vals = [per[m].loc[primary, metric].to_numpy() for m in order]
    defined = [m for m, v in zip(order, vals) if np.isfinite(v).any()]
    data = [v[np.isfinite(v)] for v in vals if np.isfinite(v).any()]
    pos = [order.index(m) + 1 for m in defined]
    ax.boxplot(data, positions=pos, showfliers=False, widths=0.6, patch_artist=True,
               boxprops=dict(facecolor="#cde2fb", edgecolor=P.BLUE), medianprops=dict(color=P.BLUE, lw=2),
               whiskerprops=dict(color=P.MUTED), capprops=dict(color=P.MUTED))
    for m in order:
        if m not in defined:
            ax.text(order.index(m) + 1, 0.02, "undefined", ha="center", va="bottom", fontsize=8, color=P.MUTED, rotation=90)
    ax.set_xticks(range(1, len(order) + 1), order, rotation=30, ha="right")
    ax.set_xlim(0.4, len(order) + 0.6)
    ax.set_title(title); ax.set_ylabel("Pearson r (primary set, per protein)")
    ax.grid(axis="x", visible=False)
fig.tight_layout()
fig.savefig(OUT_FIG / "phase3_models_val.png")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9.5, 4.2))
for ax, metric, title in zip(axes, ["pearson_all", "within_w"], ["all validation cells", "within cell type (weighted)"]):
    x, y = own_p[metric], rid_p[metric]
    lo = min(np.nanmin(x), np.nanmin(y), 0) - 0.05
    ax.plot([lo, 1], [lo, 1], color=P.MUTED, ls="--", lw=1)
    ax.scatter(x, y, s=18, c=P.BLUE, edgecolors=P.SURFACE, linewidths=0.6)
    ax.set_xlabel("own gene, Pearson r"); ax.set_ylabel(f"{best_ridge}, Pearson r")
    ax.set_title(title)
fig.suptitle("Per protein (primary set): multi-gene ridge vs. own gene", fontsize=10)
fig.tight_layout()
fig.savefig(OUT_FIG / "phase3_ridge_vs_own_gene.png")

In [ ]:
if SUBSET is None:
    np.savez_compressed(PROCESSED_DIR / "phase3_val_predictions.npz", proteins=np.array(proteins),
                        y_val=Y_va.astype(np.float32), **{k.replace(" ", "_"): v.astype(np.float32) for k, v in pred.items()})
import resource, platform
peak = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
print(f"peak memory: {(peak / 1e9 if platform.system() == 'Darwin' else peak * 1024 / 1e9):.2f} GB")